# LHRM Baseline - DR2 Control Baseline (Variant B)

**Run ID:** `RUN_DR2_B`
**Architecture:** Variant B (Standard BERT Multi-Label Baseline)
**Epochs:** 15

This is a standard multi-label BERT baseline (`[CLS] -> Linear(54)`) using BCEWithLogitsLoss. Test set is NEVER loaded or evaluated during this training run. No unapproved class balancing or hyperparameter tuning is used.

In [ ]:
# ============================================================
# Cell 1 - Environment check
# ============================================================
import subprocess, sys, os, json, hashlib, time

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    return r.returncode

print('=== GPU ===')
sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No CUDA device - training requires GPU')


In [ ]:
# ============================================================
# Cell 2 - Bundle extraction and provenance verification
# ============================================================
import zipfile, glob

BUNDLE_SEARCH = [
    '/kaggle/input/lhrm-bundle/*.zip',
    '/kaggle/working/lhrm_bundle*.zip',
]
bundle_path = None
for pattern in BUNDLE_SEARCH:
    matches = glob.glob(pattern)
    if matches:
        bundle_path = matches[0]
        break
if bundle_path is None:
    raise FileNotFoundError('Bundle ZIP not found.')
print('Bundle:', bundle_path)

WORK = '/kaggle/working'
if not os.path.isfile(os.path.join(WORK, 'run_baseline.py')):
    with zipfile.ZipFile(bundle_path, 'r') as zf:
        zf.extractall(WORK)
    print('Extracted to:', WORK)

PROJ = WORK
sys.path.insert(0, os.path.join(PROJ, 'src'))

BERT_SEARCH = [
    '/kaggle/input/bert-base-uncased',
    '/kaggle/input/google-bert-base-uncased',
    '/kaggle/input/bertbaseuncased',
]
BERT_PATH = None
for p in BERT_SEARCH:
    if os.path.isdir(p):
        BERT_PATH = p
        break
if BERT_PATH is None:
    raise FileNotFoundError('Official bert-base-uncased not found.')
print('BERT path:', BERT_PATH)

from transformers import BertTokenizer
tok = BertTokenizer.from_pretrained(BERT_PATH)
assert tok.vocab_size == 30522

AAPD_SEARCH = [
    '/kaggle/input/aapd-dataset/AAPD',
    '/kaggle/input/aapd/AAPD',
    '/kaggle/input/aapd',
]
AAPD_DIR = None
for p in AAPD_SEARCH:
    if os.path.isfile(os.path.join(p, 'train.csv')):
        AAPD_DIR = p
        break
if AAPD_DIR is None:
    raise FileNotFoundError('AAPD dataset not found.')
print('AAPD dir:', AAPD_DIR)

os.environ['LHRM_AAPD_DIR'] = AAPD_DIR
sh(f'python {PROJ}/run_baseline.py prepare --config {PROJ}/configs/final/aapd_baseline_dr2.yaml')
print('\n=== PROVENANCE VERIFIED ===')


In [ ]:
# ============================================================
# Cell 3 - Launch DR2 Train (Variant B, 15 epochs)
# ============================================================
import subprocess, sys, os

PROJ = '/kaggle/working'
RUN_ID = 'RUN_DR2_B'
CONFIG = os.path.join(PROJ, 'configs', 'final', 'aapd_baseline_dr2.yaml')

cmd = [
    'python', '-u',
    os.path.join(PROJ, 'run_baseline.py'),
    'train',
    '--config', CONFIG,
    '--run_id', RUN_ID,
    '--variant', 'B'
]

os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

print('Command:', ' '.join(cmd))
print('Launching full baseline training...\n')

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=PROJ
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError('Run failed with code ' + str(proc.returncode))


In [ ]:
# ============================================================
# Cell 4 - Package evidence ZIP (no checkpoints)
# ============================================================
import zipfile, os, hashlib, time

PROJ = '/kaggle/working'
RUN_ID = 'RUN_DR2_B'
EXP_DIR = os.path.join(PROJ, 'experiments', RUN_ID)
OUT_ZIP = f'/kaggle/working/{RUN_ID}_EVIDENCE_{time.strftime("%Y%m%d_%H%M%S")}.zip'

def walk_files(directory, exclude_exts):
    for root, _, files in os.walk(directory):
        for f in files:
            if not any(f.endswith(ext) for ext in exclude_exts):
                yield os.path.join(root, f)

if os.path.exists(EXP_DIR):
    with zipfile.ZipFile(OUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fpath in walk_files(EXP_DIR, ['.pt', '.bin']):  # exclude checkpoints
            arcname = os.path.relpath(fpath, os.path.dirname(EXP_DIR))
            zf.write(fpath, arcname)
            print('Added:', arcname)
            
    h = hashlib.sha256(open(OUT_ZIP, 'rb').read()).hexdigest()
    print(f'\nEvidence ZIP: {OUT_ZIP}')
    print(f'SHA256: {h}')
    print(f'Size: {os.path.getsize(OUT_ZIP) / 1e6:.2f} MB')
    print('\nDownload this ZIP from the Kaggle output panel.')
else:
    print('Experiment directory not found.')
